In [ ]:
# !pip install psycopg2-binary -q

In [ ]:
import pandas as pd 
import polars as pl 
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace, current_run
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc

# Update Process for DSE Surv Complete Table

This section describes the **process used to update the master table** that aggregates epidemiological data by year, disease, and health zone.

---

### 1. Ensure Master Table Exists

- The script first **creates the master table** (`target_table`) if it does not exist.
- The table contains columns for:
  - Health zone (`ZONE`, `zone_id_DHIS2`)
  - Province (`PROVINCE`, `province_id_DHIS2`)
  - Epidemiological data (`MALADIE`, `NUMSEM`, `Cas`, `Deces`, `ALERTE`, `Date`)
  - Age-specific counts (`Cas_Age`, `Deces_Age`)
- The table is **partitioned by year** to optimize queries and updates.


### 2. Partition Update Process for the Master Table

The master table is **partitioned by year**. Updating the data involves handling each year individually:

1. **Check if the year partition exists**:
   - If it **exists**:
     - Truncate the partition to remove old data
     - Insert new data from the source table for that year
   - If it **does not exist**:
     - Create a new partition for the year
     - Insert data from the source table for that year  
  
2. **Data insertion**:
   - Only rows matching the target year are inserted into the partition
   - This ensures that each partition contains only the relevant year’s data

3. **Memory management**:
   - Garbage collection (`gc.collect()`) is performed after each year's update to free memory

4. **Indexes**:
   - After all partitions are updated, indexes are created on key columns (`NUMSEM`, `PROVINCE`, `ZONE`, `MALADIE`) to speed up queries

In [ ]:
# helper functions

def check_partition_exist(engine, table: str, year: int) -> bool:
    with engine.begin() as conn:
        exists = conn.execute(text(f'''SELECT 1 FROM pg_class c 
                JOIN pg_inherits i ON c.oid = i.inhrelid 
                JOIN pg_class p ON p.oid = i.inhparent
                WHERE p.relname = '{table}' AND c.relname = '{table}_{year}';''')).scalar()
    return True if exists else False


# log msg to OH
def log_msg(msg, level: str = "info") -> None:
    if "current_run" in globals() and current_run:
        if level == "info":
            current_run.log_info(msg)
        if level == "warning":
            current_run.log_warning(msg)
        if level == "error":
            current_run.log_error(msg)
    print(f"{level} : {msg}")

### Start processing

In [ ]:
# target table name
source_table = "DSE_Surv_Epi_Complete"
target_table = "DSE_Surv_Epi_Complete_agg" 

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# Load available years
years_available = pd.read_sql(text(f'SELECT distinct "year" FROM public."{source_table}"'), engine).year.tolist()
years_available = [int(y) for y in sorted(years_available)] 
print(f"Years available in source: {years_available}")

**Select years to process (last year)**  

The years to be refreshed are selected and stored in the variable _year_targets_  
If needed, the selection of years can be changed here to include previous years.

In [ ]:
# choose the years to update
year_targets = years_available[-1:]  # Select LAST years!
log_msg(f"Aggregating '{target_table}' for years: {year_targets}")

In [ ]:
# Create the master table if it doesn't exist
with engine.begin() as conn:
    conn.execute(text(f'''
        CREATE TABLE IF NOT EXISTS public."{target_table}" (
            "zone_id_DHIS2" TEXT,
            "year" DOUBLE PRECISION,
            "MALADIE" TEXT,
            "NUMSEM" INTEGER,
            "Cas_Age" TEXT,
            "Cas" INTEGER,
            "Deces_Age" TEXT,
            "Deces" INTEGER,
            "ALERTE" DOUBLE PRECISION,
            "ZONE" TEXT,
            "PROVINCE" TEXT,
            "province_id_DHIS2" TEXT,
            "Date" DATE
        )
        PARTITION BY LIST ("year");
    '''))
print(f"Master table '{target_table}' ensured.")


In [ ]:
for year_target in year_targets:
        
    log_msg(f"Loading data from '{source_table}' for year {year_target}")
   
    # Partition name
    partition_name = f"{target_table}_{year_target}"

    # handle partition update
    if check_partition_exist(engine, target_table, year_target):
        with engine.begin() as conn:
            try:
                print(f"Partition for year {year_target} already exist, updating (truncate/insert) data..")
                conn.execute(text(f'TRUNCATE TABLE public."{partition_name}";'))
                conn.execute(text(f'INSERT INTO public."{partition_name}" SELECT * FROM public."{source_table}" WHERE "year" = {year_target};'))                
            except Exception as e:
                print(f"Error while updating the partition '{partition_name}'. {e}")
                raise 
    else:
        with engine.begin() as conn:
            try:
                print(f"Creating new partition '{partition_name}'..")
                conn.execute(text(f'CREATE TABLE public."{partition_name}" PARTITION OF public."{target_table}" FOR VALUES IN ({year_target});'))
                conn.execute(text(f'INSERT INTO public."{partition_name}" SELECT * FROM public."{source_table}" WHERE "year" = {year_target};'))                 
            except Exception as e:
                print(f"Error while creating the partition '{partition_name}'. {e}")
                raise 
    
    # clean    
    collected = gc.collect()
    print(collected)

In [ ]:
log_msg("Creating indexes..")
with engine.begin() as conn:
    for year in year_targets:
        table_name = f'{target_table}_{year}'
        index_name = f'idx_{target_table}_{year}_filters'
        print(index_name)
        sql = text(f'''CREATE INDEX IF NOT EXISTS "{index_name}" ON public."{table_name}" ("NUMSEM",  "PROVINCE", "ZONE", "MALADIE");''')
        conn.execute(sql)

In [ ]:
log_msg(f"Table {target_table} updated!")